# Training Optimization – Thực nghiệm 2
## Khảo sát Learning Rate và Data Augmentation

Mục tiêu: giữ nguyên kiến trúc `EmotionCNN`, dữ liệu, seed và số epoch; chỉ thay đổi **learning rate** và việc bật/tắt **data augmentation**. Mô hình tốt nhất được chọn theo `PublicTest loss`; `PrivateTest` không được dùng để lựa chọn cấu hình.

# 1. Data Processing
## 1.1. Data Load

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.preprocessing import LabelEncoder

DATASET_DIR = Path('BTLON02') / 'DATA'
OUTPUT_DIR = Path('results') / 'data_processing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLASSES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

def get_files(folder, prefix):
    return sorted(
        path for label in CLASSES
        for path in (DATASET_DIR / folder / label).glob(f'{prefix}*.jpg')
    )

def load_images(files):
    images = []
    for file in files:
        with Image.open(file) as image:
            array = np.asarray(image.convert('L'), dtype=np.uint8)
        if array.shape != (48, 48):
            raise ValueError(f'Invalid image size {array.shape}: {file}')
        images.append(array[..., np.newaxis])
    return np.asarray(images)

train_files = get_files('train', 'Training_')
public_files = get_files('test', 'PublicTest_')
private_files = get_files('test', 'PrivateTest_')

train_labels = [file.parent.name for file in train_files]
public_labels = [file.parent.name for file in public_files]
private_labels = [file.parent.name for file in private_files]

train_images = load_images(train_files)
public_images = load_images(public_files)
private_images = load_images(private_files)

Error: No connection selected.

## 1.2. Convert to One-Hot Encoding

In [2]:
label_encoder = LabelEncoder()
label_encoder.fit(train_labels + public_labels + private_labels)
classes = label_encoder.classes_

def to_one_hot(labels):
    encoded = label_encoder.transform(labels)
    return np.eye(len(classes), dtype=np.float32)[encoded]

x_train = train_images
y_train = to_one_hot(train_labels)
x_valid = public_images
y_valid = to_one_hot(public_labels)
x_test = private_images
y_test = to_one_hot(private_labels)

print(f'Training: {len(x_train)} | Public: {len(x_valid)} | Private: {len(x_test)}')
print(f'Image information: {x_train.shape[1]} x {x_train.shape[2]} x {x_train.shape[3]}')
print(f'Classes: {classes.tolist()}')
print(f'One-hot shapes: train={y_train.shape}, valid={y_valid.shape}, test={y_test.shape}')

Error: No connection selected.

## 1.3. Data Explorer
### 1.3.1. Histogram of Dataset Distribution

In [ ]:
def plot_histogram(labels, title, axis):
    counts = [labels.count(label) for label in CLASSES]
    axis.bar(CLASSES, counts)
    axis.set_title(title)
    axis.set_ylabel('Number of images')
    axis.tick_params(axis='x', rotation=35)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_histogram(train_labels, 'Training Dataset', axes[0])
plot_histogram(public_labels, 'PublicTest Dataset', axes[1])
plot_histogram(private_labels, 'PrivateTest Dataset', axes[2])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'histogram_data.png', dpi=150, bbox_inches='tight')
plt.show()

![Histogram of Dataset Distribution](results/data_processing/histogram_data.png)

### 1.3.2. Training / Testing Image Samples

In [ ]:
def show_samples(images, one_hot_labels, title, filename, seed=42):
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(images), size=32, replace=False)
    fig, axes = plt.subplots(4, 8, figsize=(14, 7))
    for axis, index in zip(axes.flat, indices):
        axis.imshow(images[index, :, :, 0], cmap='gray')
        axis.set_title(classes[one_hot_labels[index].argmax()])
        axis.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=150, bbox_inches='tight')
    plt.show()

show_samples(x_train, y_train, 'Training Images', 'training_images.png')
show_samples(x_valid, y_valid, 'Validation Images', 'validating_images.png')

![Training Images](results/data_processing/training_images.png)

![Validation Images](results/data_processing/validating_images.png)

## 2.1. Data augmentation

Bản PyTorch tương đương cấu hình tài liệu: `fill_mode='nearest'`, lật ngang, xoay ±10°, dịch ngang/dọc tối đa 10% và zoom 0.9–1.1. Không dùng ZCA whitening, featurewise centering/standardization hay vertical flip. Các biến đổi chỉ áp dụng cho Training; PublicTest và PrivateTest luôn chỉ grayscale → resize 48×48 → tensor [0, 1].

In [7]:
from __future__ import annotations

import csv
import json
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from data_loader import CLASSES, DATA_DIR, scan_dataset, seed_everything
from model import EmotionCNN, save_checkpoint

SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATES = (1e-3, 1e-4, 1e-5)
AUGMENTATION_OPTIONS = (False, True)
OUTPUT_DIR = Path('results') / 'experiment_2'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'

# Tương đương các tham số ImageDataGenerator trong tài liệu.
AUGMENTATION_CONFIG = {
    'fill_mode': 'nearest', 'horizontal_flip': True, 'rotation_range': 10.0,
    'width_shift_range': 0.1, 'height_shift_range': 0.1, 'zoom_range': 0.1,
    'vertical_flip': False, 'zca_whitening': False,
    'featurewise_center': False, 'featurewise_std_normalization': False,
}

seed_everything(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')
print(json.dumps(AUGMENTATION_CONFIG, indent=2))

Error: No connection selected.

In [8]:
def build_experiment_transform(augment=False):
    steps = [transforms.Grayscale(1), transforms.Resize((48, 48))]
    if augment:
        steps += [
            transforms.RandomHorizontalFlip(),
            transforms.RandomAffine(
                degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1),
                interpolation=transforms.InterpolationMode.BILINEAR, fill=0,
            ),
        ]
    return transforms.Compose(steps + [transforms.ToTensor()])

class FERExperimentDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples, self.transform = list(samples), build_experiment_transform(augment)
    def __len__(self): return len(self.samples)
    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            return self.transform(ImageOps.exif_transpose(image)), label

samples = scan_dataset(DATA_DIR)
print({split: len(items) for split, items in samples.items()})

def make_loaders(augment):
    train_set = FERExperimentDataset(samples['train'], augment=augment)
    valid_set = FERExperimentDataset(samples['valid'], augment=False)
    generator = torch.Generator().manual_seed(SEED)
    return (
        DataLoader(train_set, BATCH_SIZE, shuffle=True, generator=generator, pin_memory=DEVICE.type == 'cuda'),
        DataLoader(valid_set, BATCH_SIZE, shuffle=False, pin_memory=DEVICE.type == 'cuda'),
    )

# Kiểm tra trực quan: ảnh gốc (trái) và một lần augmentation (phải).
image_path, label = samples['train'][0]
with Image.open(image_path) as image: original = build_experiment_transform(False)(ImageOps.exif_transpose(image)).squeeze()
with Image.open(image_path) as image: augmented = build_experiment_transform(True)(ImageOps.exif_transpose(image)).squeeze()
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, tensor, name in zip(axes, (original, augmented), ('Original', 'Augmented')):
    ax.imshow(tensor, cmap='gray'); ax.set_title(f'{name}: {CLASSES[label]}'); ax.axis('off')
plt.tight_layout()

Error: No connection selected.

## 2.2. Training model

Notebook dùng `EmotionCNN` trong `model.py` để nhất quán với chương trình dự đoán hiện tại. Kiến trúc được giữ cố định giữa mọi lượt chạy; vì vậy chênh lệch kết quả chỉ có thể đến từ learning rate hoặc augmentation. Đây **không phải** bản sao MEDIUM_01 Keras (3/4/4 convolution blocks) trong tài liệu tham khảo.

In [ ]:
model = EmotionCNN(len(CLASSES)).to(DEVICE)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
del model

Error: No connection selected.

## 2.3. Callbacks tương đương

Mỗi cấu hình có: checkpoint tốt nhất theo `val_loss`, CSV log, early stopping với patience 10 và dừng rõ ràng nếu loss/gradient không hữu hạn. Không bật `ReduceLROnPlateau`, vì thay đổi LR trong lúc train sẽ làm nhiễu phép khảo sát learning rate ban đầu.

In [10]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = correct = count = 0
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if training: optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if not torch.isfinite(loss):
                raise FloatingPointError('Loss NaN/Inf; run stopped without saving an invalid checkpoint.')
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0, error_if_nonfinite=True)
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            count += labels.size(0)
    return total_loss / count, correct / count

def run_experiment(learning_rate, augment):
    tag = f'lr_{learning_rate:.0e}_aug_{int(augment)}'
    seed_everything(SEED)
    train_loader, valid_loader = make_loaders(augment)
    model = EmotionCNN(len(CLASSES)).to(DEVICE)
    optimizer, criterion = Adam(model.parameters(), lr=learning_rate), nn.CrossEntropyLoss()
    checkpoint_path, csv_path = CHECKPOINT_DIR / f'{tag}.pth', OUTPUT_DIR / f'{tag}.csv'
    history, best_loss, stale = [], float('inf'), 0
    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.perf_counter()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, valid_loader, criterion)
        row = dict(epoch=epoch, train_loss=train_loss, train_accuracy=train_acc, val_loss=val_loss, val_accuracy=val_acc, learning_rate=learning_rate, seconds=time.perf_counter()-started)
        history.append(row)
        if val_loss < best_loss:
            best_loss, stale = val_loss, 0
            save_checkpoint(model, checkpoint_path, epoch=epoch, validation_loss=val_loss, validation_accuracy=val_acc, history=history, config={'learning_rate': learning_rate, 'augmentation': augment, 'seed': SEED}, train_size=len(train_loader.dataset), valid_size=len(valid_loader.dataset))
        else: stale += 1
        print(f'{tag} | epoch {epoch:02d} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')
        if stale >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=history[0].keys()); writer.writeheader(); writer.writerows(history)
    best = min(history, key=lambda row: row['val_loss'])
    return {**best, 'augmentation': augment, 'checkpoint': str(checkpoint_path), 'log': str(csv_path), 'epochs_ran': len(history)}

Error: No connection selected.

## 2.4. Results training

Chạy cell dưới để thực hiện 6 lượt: 3 LR × 2 trạng thái augmentation. Mỗi lượt có seed giống nhau, mô hình khởi tạo lại từ đầu và lưu riêng checkpoint/log. Lần chạy toàn bộ có thể mất khá lâu trên CPU.

In [11]:
results = []
for learning_rate in LEARNING_RATES:
    for augment in AUGMENTATION_OPTIONS:
        results.append(run_experiment(learning_rate, augment))

with (OUTPUT_DIR / 'summary.json').open('w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
results

Error: No connection selected.

In [12]:
# Chạy sau cell khảo sát. Sắp xếp theo val_loss tăng dần để chọn cấu hình.
results = sorted(results, key=lambda row: row['val_loss'])
for row in results:
    print(f"lr={row['learning_rate']:.0e} | augmentation={row['augmentation']} | best epoch={row['epoch']} | val_loss={row['val_loss']:.4f} | val_acc={row['val_accuracy']:.4f}")

labels = [f"{r['learning_rate']:.0e}\naug={r['augmentation']}" for r in results]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(labels, [r['val_loss'] for r in results]); axes[0].set_title('Best PublicTest loss (lower is better)'); axes[0].set_ylabel('Loss')
axes[1].bar(labels, [r['val_accuracy'] for r in results]); axes[1].set_title('PublicTest accuracy at best loss'); axes[1].set_ylabel('Accuracy')
for ax in axes: ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'comparison.png', dpi=150)
best = results[0]
print(f"Kết luận: chọn LR={best['learning_rate']:.0e}, augmentation={best['augmentation']} (PublicTest loss={best['val_loss']:.4f}, accuracy={best['val_accuracy']:.4f}).")

Error: No connection selected.

### Mẫu báo cáo kết luận

Sau khi chạy, điền số liệu từ `summary.json` vào bảng so sánh và nêu: (1) LR nào cho `PublicTest loss` thấp nhất, (2) augmentation có cải thiện tổng quát hóa không, (3) cấu hình được chọn. Chỉ đánh giá một lần trên `PrivateTest` sau khi đã chốt cấu hình, để tránh rò rỉ tập test.

# 3. Evaluations

Run this section only after selecting `best` using PublicTest. It evaluates the selected checkpoint exactly once on PrivateTest and produces the five report figures: data histogram, validating images, confusion matrix, wrong cases, and true cases.

In [ ]:
# `best` is created by the comparison cell; PrivateTest is never used for selection.
if 'best' not in globals():
    raise RuntimeError('Run the comparison cell first so that `best` is available.')

private_set = FERExperimentDataset(samples['test'], augment=False)
private_loader = DataLoader(private_set, BATCH_SIZE, shuffle=False, pin_memory=DEVICE.type == 'cuda')
checkpoint = torch.load(best['checkpoint'], map_location=DEVICE)
final_model = EmotionCNN(len(CLASSES)).to(DEVICE)
final_model.load_state_dict(checkpoint['model_state_dict'])
final_model.eval()
print(f'PrivateTest images: {len(private_set):,}')
print('Image information: 48 x 48 x 1')

In [ ]:
# 3.1 Histogram of Evaluation Data
test_counts = Counter(label for _, label in samples['test'])
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(CLASSES, [test_counts[i] for i in range(len(CLASSES))], color='#4c72b0')
ax.bar_label(bars, padding=3)
ax.set(title='Histogram of Evaluation Dataset', ylabel='Number of images')
ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'evaluation_histogram.png', dpi=150); plt.show()

# 3.2 Validating Images: deterministic sample from PrivateTest with true labels.
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(private_set), generator=generator)[:32].tolist()
fig, axes = plt.subplots(4, 8, figsize=(14, 8))
for ax, index in zip(axes.flat, indices):
    image, label = private_set[index]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(CLASSES[label], fontsize=9); ax.axis('off')
fig.suptitle('Validating Images (True Labels)', y=1.01)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'evaluation_samples.png', dpi=150); plt.show()

In [ ]:
# Predict each PrivateTest image once and retain results for sections 3.3--3.5.
all_images, true_labels, predicted_labels = [], [], []
with torch.inference_mode():
    for images, labels in private_loader:
        predictions = final_model(images.to(DEVICE)).argmax(dim=1).cpu()
        all_images.append(images.cpu()); true_labels.append(labels.cpu()); predicted_labels.append(predictions)
all_images = torch.cat(all_images)
true_labels = torch.cat(true_labels)
predicted_labels = torch.cat(predicted_labels)
final_accuracy = (predicted_labels == true_labels).float().mean().item()
print(f'PrivateTest average accuracy: {final_accuracy:.2%}')

# 3.3 Confusion Matrix: every row is normalised by its true-label total.
matrix = torch.zeros((len(CLASSES), len(CLASSES)), dtype=torch.int64)
for actual, predicted in zip(true_labels, predicted_labels):
    matrix[actual, predicted] += 1
matrix_percent = matrix.float() / matrix.sum(dim=1, keepdim=True).clamp_min(1) * 100
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(matrix_percent, cmap='Blues', vmin=0, vmax=100)
fig.colorbar(image, ax=ax, label='Percent')
ax.set(xticks=range(len(CLASSES)), yticks=range(len(CLASSES)), xticklabels=CLASSES, yticklabels=CLASSES, xlabel='Predicted label', ylabel='True label', title=f'Confusion Matrix - Average Accuracy {final_accuracy:.2%}')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
for row in range(len(CLASSES)):
    for col in range(len(CLASSES)):
        value = matrix_percent[row, col].item()
        ax.text(col, row, f'{value:.0f}%', ha='center', va='center', color='white' if value > 50 else '#222')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'confusion_matrix_private_test.png', dpi=150); plt.show()

In [ ]:
def show_prediction_cases(mask, title, filename, limit=32):
    indices = torch.where(mask)[0][:limit].tolist()
    fig, axes = plt.subplots(4, 8, figsize=(14, 8))
    for ax in axes.flat:
        ax.axis('off')
    for ax, index in zip(axes.flat, indices):
        ax.imshow(all_images[index].squeeze(), cmap='gray')
        ax.set_title(f'{CLASSES[predicted_labels[index]]} ({CLASSES[true_labels[index]]})', fontsize=8)
        ax.axis('off')
    fig.suptitle(title + ' (Predict / True)', y=1.01)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / filename, dpi=150); plt.show()

# 3.4 Wrong Cases and 3.5 True Cases.
show_prediction_cases(predicted_labels != true_labels, 'Wrong Predict Images', 'wrong_cases.png')
show_prediction_cases(predicted_labels == true_labels, 'True Predict Images', 'true_cases.png')